In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import dask
import numpy as np
import pandas as pd
from dagster_components.utils import cast_all_columns_to_numeric

from nu_segregation.defs.assets.prod import get_seg_full

In [3]:
df_survey = pd.DataFrame(pd.read_parquet("./survey.parquet").dropna(subset=["Ingreso"]))
df_census = pd.DataFrame(
    pd.read_parquet("./census.parquet")
    .set_index("cvegeo")
    .pipe(cast_all_columns_to_numeric, make_valid_int=True)
)

In [38]:
df_census

,p_15ymas,ConexionInt_internet,ConexionInt_no_internet,Edad_p15_64,Edad_p65mas,EstadoConyu_casada,EstadoConyu_separada,EstadoConyu_soltera,Nivel_ninguno,Nivel_posbasica,Nivel_primaria,Nivel_secundaria,Sexo_f,Sexo_m
cvegeo,,,,,,,,,,,,,,
0100100014448,1799,1069,730,1775,24,1295,164,340,42,918,181,658,915,884
0100100013721,3372,1714,1658,3308,64,2240,350,782,188,1134,538,1512,1727,1645
0100100014255,3371,2234,1137,3292,79,2343,261,767,48,2429,170,724,1707,1664
010010001401A,3795,3563,232,3648,147,2553,231,1011,21,3558,57,159,1915,1880
0100100014503,1430,1352,78,1333,97,935,113,382,23,1268,41,98,726,704
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0100100014467,1552,467,1085,1534,18,1001,195,356,162,359,313,718,823,729
0100100010017,1460,425,1035,1437,23,958,179,323,81,424,253,702,763,697
0100100014433,5760,2437,3323,5669,91,3891,619,1250,372,2050,956,2382,3042,2718


In [35]:
test = get_seg_full(df_census, df_survey, out_path="./results/")

DataSourceError: '/vsizip/data/agebs.zip' does not exist in the file system, and is not recognized as a supported dataset name.

In [36]:
df_census

,p_15ymas,ConexionInt_internet,ConexionInt_no_internet,Edad_p15_64,Edad_p65mas,EstadoConyu_casada,EstadoConyu_separada,EstadoConyu_soltera,Nivel_ninguno,Nivel_posbasica,Nivel_primaria,Nivel_secundaria,Sexo_f,Sexo_m
cvegeo,,,,,,,,,,,,,,
0100100014448,1799,1069,730,1775,24,1295,164,340,42,918,181,658,915,884
0100100013721,3372,1714,1658,3308,64,2240,350,782,188,1134,538,1512,1727,1645
0100100014255,3371,2234,1137,3292,79,2343,261,767,48,2429,170,724,1707,1664
010010001401A,3795,3563,232,3648,147,2553,231,1011,21,3558,57,159,1915,1880
0100100014503,1430,1352,78,1333,97,935,113,382,23,1268,41,98,726,704
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0100100014467,1552,467,1085,1534,18,1001,195,356,162,359,313,718,823,729
0100100010017,1460,425,1035,1437,23,958,179,323,81,424,253,702,763,697
0100100014433,5760,2437,3323,5669,91,3891,619,1250,372,2050,956,2382,3042,2718


In [32]:
res

Ingreso,1.0,2.0,3.0,4.0,5.0,total_ipf
cvegeo,,,,,,
0100100010017,332.533454,313.662844,338.138680,282.796995,192.868026,1460.0
010010001006A,165.122912,162.158881,159.409755,229.615251,340.693201,1057.0
0100100010106,332.914566,334.436855,339.785894,464.577081,631.285604,2103.0
0100100010163,298.094636,292.941707,313.345715,359.765452,401.852491,1666.0
0100100010182,258.399027,261.435359,257.423678,387.959396,598.782540,1764.0
...,...,...,...,...,...,...
0101100520189,156.003981,150.717807,165.381464,146.667250,109.229498,728.0
0101101280136,542.245918,516.838342,543.165424,546.585603,524.164713,2673.0
0101101380210,266.856509,254.859535,272.816621,250.646083,208.821251,1254.0


In [6]:
n_samples = 32
n_ind = len(df_survey)

rng = np.random.default_rng(seed=42)
bs_idxs = rng.integers(0, n_ind, size=(n_samples, n_ind))

get_seg_full_delayed = dask.delayed(get_seg_full)

In [ ]:
temp = df_survey.iloc[bs_idxs[0]].reset_index(drop=True)
res = get_seg_full_delayed(
    df_census,
    temp,
)